In [2]:
import random
import string
import timeit

# Implementation

In [3]:
def rabin_karp(text, pattern, d=256, q=101):
    """
    Rabin-Karp algorithm for pattern searching.
    
    Args:
        text: The text to search in
        pattern: The pattern to search for
        d: Number of characters in the input alphabet (default 256 for ASCII)
        q: A prime number for hashing (default 101)
    
    Returns:
        List of starting indices where pattern is found in text
    """
    n = len(text)
    m = len(pattern)
    matches = []
    
    if m > n:
        return matches
    
    # Calculate hash value for pattern and first window of text
    p = 0  # hash value for pattern
    t = 0  # hash value for text window
    h = 1  # d^(m-1) % q
    
    # Calculate h = d^(m-1) % q
    for i in range(m - 1):
        h = (h * d) % q
    
    # Calculate initial hash values
    for i in range(m):
        p = (d * p + ord(pattern[i])) % q
        t = (d * t + ord(text[i])) % q
    
    # Slide the pattern over text
    for i in range(n - m + 1):
        # Check if hash values match
        if p == t:
            # If hash matches, check character by character
            if text[i:i + m] == pattern:
                matches.append(i)
        
        # Calculate hash for next window (if not last iteration)
        if i < n - m:
            # Remove leading character and add trailing character
            t = (d * (t - ord(text[i]) * h) + ord(text[i + m])) % q
            
            # Handle negative hash values
            if t < 0:
                t += q
    
    return matches

In [4]:
if __name__ == "__main__":
    text = "AABAACAADAABAABA"
    pattern = "AABA"
    
    print(f"Text: {text}")
    print(f"Pattern: {pattern}")
    
    result = rabin_karp(text, pattern)
    
    if result:
        print(f"\nPattern found at indices: {result}")
        for idx in result:
            print(f"  Index {idx}: {text[idx:idx + len(pattern)]}")
    else:
        print("\nPattern not found in text")
    
    print("\n" + "="*50)
    print("Additional test cases:")
    
    test_cases = [
        ("hello world hello", "hello"),
        ("abcdefg", "xyz"),
        ("ababcababa", "aba"),
        ("AAAA", "AA")
    ]
    
    for txt, pat in test_cases:
        result = rabin_karp(txt, pat)
        print(f"\nText: '{txt}'\nPattern: '{pat}'\nMatches at: {result}")

Text: AABAACAADAABAABA
Pattern: AABA

Pattern found at indices: [0, 9, 12]
  Index 0: AABA
  Index 9: AABA
  Index 12: AABA

Additional test cases:

Text: 'hello world hello'
Pattern: 'hello'
Matches at: [0, 12]

Text: 'abcdefg'
Pattern: 'xyz'
Matches at: []

Text: 'ababcababa'
Pattern: 'aba'
Matches at: [0, 5, 7]

Text: 'AAAA'
Pattern: 'AA'
Matches at: [0, 1, 2]


## Benchmarking

In [5]:
def random_string(n, alphabet=string.ascii_lowercase):
    return ''.join(random.choice(alphabet) for _ in range(n))

def benchmark(func, text, pattern, repeat=5):
    """
    Benchmarks func(text, pattern) over several repetitions.
    Returns average runtime in seconds.
    """
    timer = timeit.Timer(lambda: func(text, pattern))
    times = timer.repeat(repeat=repeat, number=1)
    return sum(times) / len(times)

In [6]:
def run_benchmarks():
    random.seed(42)

    text_sizes = [10_000, 50_000, 100_000, 200_000]
    pattern_length = 20

    print(f"{'Text Size':>12} | {'RK Time (s)':>12} | {'find() Time (s)':>15}")
    print("-" * 45)

    for n in text_sizes:
        text = random_string(n)
        start = random.randint(0, n - pattern_length)
        pattern = text[start:start + pattern_length]

        rk_time = benchmark(rabin_karp, text, pattern)
        builtin_time = benchmark(lambda t, p: t.find(p), text, pattern)

        print(f"{n:12d} | {rk_time:12.6f} | {builtin_time:15.6f}")

if __name__ == "__main__":
    run_benchmarks()

   Text Size |  RK Time (s) | find() Time (s)
---------------------------------------------
       10000 |     0.008960 |        0.000008
       50000 |     0.015795 |        0.000015
      100000 |     0.043204 |        0.000077
      200000 |     0.069526 |        0.000031
